In [2]:
import os
from pathlib import Path
from dotenv import load_dotenv

# Explicitly load the .env next to this notebook
env_path = Path(r"c:\Users\DanielTaylor\StoreInsight\store_insight\pipeline_ui\backend\.env")
print(f".env exists: {env_path.exists()}")
print(f".env contents (keys only):")
if env_path.exists():
    for line in env_path.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            key = line.split("=", 1)[0]
            val = line.split("=", 1)[1]
            # Mask sensitive values
            if "KEY" in key or "PASSWORD" in key or "PASS" in key:
                print(f"  {key} = ***SET***")
            else:
                print(f"  {key} = {val}")

load_dotenv(env_path, override=True)

print("\n--- Resolved env vars ---")
print("SNOWFLAKE_ACCOUNT:", repr(os.getenv('SNOWFLAKE_ACCOUNT')))
print("SNOWFLAKE_USER:", repr(os.getenv('SNOWFLAKE_USER')))
print("SNOWFLAKE_ROLE:", repr(os.getenv('SNOWFLAKE_ROLE')))
print("SNOWFLAKE_WAREHOUSE:", repr(os.getenv('SNOWFLAKE_WAREHOUSE')))
print("SNOWFLAKE_DATABASE:", repr(os.getenv('SNOWFLAKE_DATABASE')))
print("SNOWFLAKE_SCHEMA:", repr(os.getenv('SNOWFLAKE_SCHEMA')))
print("SNOWFLAKE_PRIVATE_KEY_PATH:", repr(os.getenv('SNOWFLAKE_PRIVATE_KEY_PATH')))
print("SNOWFLAKE_PASSWORD:", 'SET' if os.getenv('SNOWFLAKE_PASSWORD') else 'NOT SET')

.env exists: True
.env contents (keys only):
  SNOWFLAKE_ACCOUNT = UUTQCJV-UKCOOP
  SNOWFLAKE_USER = daniel.t@retailinsight.io
  SNOWFLAKE_ROLE = ACL_SNOWFLAKE_RI_EOE_PRD_DP_USER
  SNOWFLAKE_WAREHOUSE = STORE_INSIGHT_WH
  SNOWFLAKE_DATABASE = STORE_INSIGHT_DB
  SNOWFLAKE_SCHEMA = PUBLIC
  ALLOWED_ORIGINS = http://localhost:8081

--- Resolved env vars ---
SNOWFLAKE_ACCOUNT: 'UUTQCJV-UKCOOP'
SNOWFLAKE_USER: 'daniel.t@retailinsight.io'
SNOWFLAKE_ROLE: 'ACL_SNOWFLAKE_RI_EOE_PRD_DP_USER'
SNOWFLAKE_WAREHOUSE: 'STORE_INSIGHT_WH'
SNOWFLAKE_DATABASE: 'STORE_INSIGHT_DB'
SNOWFLAKE_SCHEMA: 'PUBLIC'
SNOWFLAKE_PRIVATE_KEY_PATH: None
SNOWFLAKE_PASSWORD: NOT SET


In [3]:
from snowflake.snowpark import Session

params = {
    "account":   os.getenv("SNOWFLAKE_ACCOUNT"),
    "user":      os.getenv("SNOWFLAKE_USER"),
    "role":      os.getenv("SNOWFLAKE_ROLE"),
    "warehouse": os.getenv("SNOWFLAKE_WAREHOUSE"),
    "database":  os.getenv("SNOWFLAKE_DATABASE"),
    "schema":    os.getenv("SNOWFLAKE_SCHEMA"),
}

# Auth: private key or password
key_path = os.getenv("SNOWFLAKE_PRIVATE_KEY_PATH")
password = os.getenv("SNOWFLAKE_PASSWORD")

if key_path:
    from cryptography.hazmat.backends import default_backend
    from cryptography.hazmat.primitives.serialization import (
        Encoding, NoEncryption, PrivateFormat, load_pem_private_key, load_der_private_key
    )
    with open(key_path, "rb") as f:
        key_data = f.read()
    passphrase = os.getenv("SNOWFLAKE_PRIVATE_KEY_PASSPHRASE")
    passphrase = passphrase.encode() if passphrase else None
    try:
        private_key = load_pem_private_key(key_data, password=passphrase, backend=default_backend())
    except ValueError:
        private_key = load_der_private_key(key_data, password=passphrase, backend=default_backend())
    params["private_key"] = private_key.private_bytes(
        encoding=Encoding.DER, format=PrivateFormat.PKCS8, encryption_algorithm=NoEncryption()
    )
    print("Auth: private key")
elif password:
    params["password"] = password
    print("Auth: password")
else:
    print("⚠️  No private key or password found — will attempt externalbrowser (may fail in notebook)")
    params["authenticator"] = "externalbrowser"

print("Connecting...")
session = Session.builder.configs(params).create()
print("✅ Connected!")
session.sql("SELECT CURRENT_ACCOUNT(), CURRENT_USER(), CURRENT_ROLE(), CURRENT_DATABASE(), CURRENT_SCHEMA()").show()

⚠️  No private key or password found — will attempt externalbrowser (may fail in notebook)
Connecting...


 pip install snowflake-connector-python[secure-local-storage]


Initiating login request with your identity provider. A browser window should have opened for you to complete the login. If you can't see it, check existing browser windows, or your OS settings. Press CTRL+C to abort and try again...
Going to open: https://login.microsoftonline.com/93a31d5d-1a5d-433f-baa0-138549cd2108/saml2?SAMLRequest=nZJPj9owEMW%2FSuSek9gJULCAFQvaLipbEH9Kuzdv4oCFY6cehwCfvk4AaXvYPfQSRfab%2BT3Pm%2F7DKZfekRsQWg0QCTDyuEp0KtRugDbrJ7%2BLPLBMpUxqxQfozAE9DPvAclnQUWn3asn%2FlBys5xopoPXFAJVGUc1AAFUs50BtQlejlxmNAkwZADfW4dCtJAXhWHtrCxqGVVUFVRxoswsjjHGIe6FT1ZIv6B2i%2BJxRGG11ouW95OTe9AGChLhVI5zCERa3wkehriP4jPJ2FQF9Xq8X%2FmK%2BWiNvdH%2FdWCsoc25W3BxFwjfL2dUAOAeQk06MW0F58EG7GQbsUhoegNJVJtmBJzovSutaB%2B4vzHgaSr0TbmDTyQAVB5Huttvk92lUzLbj7vmYvSzV7Lmz3y%2FyX6%2BXy2POvsnvZjM%2Fqed5lSDv5z3eqI53ClDyqapDte4IRx0ff%2FWj9poQ2o4oiYN21HpF3sSFKhSzTeXdeeMjyEViNOjMaiWF4o3LXsxikrZTnzD3acVx5r8xhn0Sd9utXpJGBHfDOroIXdeHNkbM8L%2BG0g%2Fft7it4w%2BX0HSy0FIkZ%2B9Jm5zZjwMkAWlOROpnjZTynAk5SlPDAVyQUupqbDizbuutKT

 pip install snowflake-connector-python[secure-local-storage]


✅ Connected!
----------------------------------------------------------------------------------------------------------------------------------
|"CURRENT_ACCOUNT()"  |"CURRENT_USER()"           |"CURRENT_ROLE()"                  |"CURRENT_DATABASE()"  |"CURRENT_SCHEMA()"  |
----------------------------------------------------------------------------------------------------------------------------------
|SM16304              |DANIEL.T@RETAILINSIGHT.IO  |ACL_SNOWFLAKE_RI_EOE_PRD_DP_USER  |STORE_INSIGHT_DB      |PUBLIC              |
----------------------------------------------------------------------------------------------------------------------------------



In [4]:
# Quick test: query the store hierarchy table
session.table("STORE_HIERARCHY_V1").select("STORE_ID", "STORE_NAME").limit(5).show()

-----------------------------
|"STORE_ID"  |"STORE_NAME"  |
-----------------------------
|455         |COBHOLM       |
|456         |Costessey     |
|457         |Catton Grove  |
|458         |Dereham Road  |
|459         |Gorleston     |
-----------------------------



In [5]:
session.close()
print("Session closed.")

Session closed.
